# Run Student Performance Prediction Project in Google Colab

This notebook runs the complete project in one session. First upload the project ZIP downloaded from this chat. The code installs dependencies, runs the pipeline, displays the generated reports, and lets you download the outputs. The first run downloads the OULAD dataset (about 47 MB compressed; the expanded clickstream CSV is much larger).

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()  # Select student-performance-risk.zip
zip_name = next(iter(uploaded))
project_dir = Path('/content/student-performance-risk')
if project_dir.exists():
    shutil.rmtree(project_dir)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
# If the ZIP contained the project folder, use it; otherwise locate run_project.py.
if not (project_dir / 'run_project.py').exists():
    candidates = list(Path('/content').glob('**/run_project.py'))
    candidates = [p for p in candidates if 'student-performance-risk' in str(p)]
    if not candidates:
        raise FileNotFoundError('Could not find run_project.py inside the uploaded ZIP.')
    project_dir = candidates[0].parent
os.chdir(project_dir)
print('Project folder:', Path.cwd())
print('Files:', [p.name for p in Path.cwd().iterdir()][:15])

## Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## Run the full project

In [ ]:
!python run_project.py

## Results: baseline and tuned models

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image

metrics = pd.read_csv('reports/model_comparison.csv')
display(metrics.sort_values('f1_at_risk', ascending=False).round(3))
display(Image(filename='reports/figures/model_comparison.png'))
display(Image(filename='reports/figures/tuned_xgboost_confusion_matrix.png'))

## Explainability

In [ ]:
importance = pd.read_csv('reports/feature_importance.csv')
display(importance.head(20))
display(Image(filename='reports/figures/shap_summary.png'))

## Read the generated reports

In [ ]:
display(Markdown(Path('reports/eda_report.md').read_text(encoding='utf-8')))
display(Markdown(Path('reports/model_evaluation_report.md').read_text(encoding='utf-8')))

## Download completed submission artifacts

The archive includes the two generated reports, figures, feature data and fitted model. It does not include the large raw dataset.

In [ ]:
import shutil
archive_path = shutil.make_archive('/content/student-performance-results', 'zip', root_dir=str(Path.cwd()), base_dir='reports')
# Add the model artifacts and processed feature table to a combined output archive.
with zipfile.ZipFile(archive_path, 'a', compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in ['artifacts/student_risk_model.joblib', 'artifacts/model_metadata.json', 'data/processed/student_features.csv', 'predict_risk.py']:
        if Path(rel).exists():
            zf.write(rel, arcname=rel)
files.download(archive_path)